In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

In [6]:
data_path = Path("../data/processed/training_data.csv")

print(data_path.exists())

True


In [7]:
training_data = pd.read_csv(data_path)

df = training_data.copy()


In [8]:
X = df.drop(columns=["action"])
y = df["action"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, {"n_estimators":[100,200], "max_depth":[6,10,None]}, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)

gb = GradientBoostingClassifier(random_state=42)
gb_grid = GridSearchCV(gb, {"n_estimators":[100,200], "learning_rate":[0.05,0.1], "max_depth":[3,5]}, cv=5, n_jobs=-1)
gb_grid.fit(X_train, y_train)

print("RF best:", rf_grid.best_score_, rf_grid.best_params_)
print("GB best:", gb_grid.best_score_, gb_grid.best_params_)

best_model = rf_grid.best_estimator_ if rf_grid.best_score_ >= gb_grid.best_score_ else gb_grid.best_estimator_
print(classification_report(y_test, best_model.predict(X_test)))

RF best: 1.0 {'max_depth': 6, 'n_estimators': 100}
GB best: 1.0 {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00       133
           2       1.00      1.00      1.00       633
           3       1.00      1.00      1.00       166

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [9]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort
import numpy as np

onnx_model = convert_sklearn(best_model, initial_types=[('float_input', FloatTensorType([None, X.shape[1]]))])
with open("../models/optigrid_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

session = ort.InferenceSession("../models/optigrid_model.onnx")
sample = X_test.iloc[[0]].values.astype(np.float32)
print("ONNX:", session.run(None, {"float_input": sample})[0])
print("sklearn:", best_model.predict(X_test.iloc[[0]]))

ONNX: [2]
sklearn: [2]
